In [ ]:
import pandas as pd

# Load the new CSV file
df = pd.read_csv("dataset_12000_records.csv")

# Look at the first 5 rows to understand the structure
display(df.head())

# Check the data types (numbers vs text)
print("\n--- Dataset Info ---")
df.info()

# Calculate the exact number of missing (NaN) values in each column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

In [62]:
from sklearn.model_selection import train_test_split

# 1. Separate what we want to predict (y) from the data we use to predict it (X)
# X gets everything EXCEPT the 'Readmission' column
X = df.drop(['Readmitted_30_Days', 'Patient_ID'], axis=1)

# y gets ONLY the 'Readmission' column
y = df['Readmitted_30_Days']

# 2. Split the data: 80% for training the model, 20% for testing it later
# random_state=42 ensures you get the exact same random split every time you run this cell
# stratify=y ensures the 80/20 split maintains the same ratio of readmitted patients
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training data shape (rows, columns):", X_train.shape)
print("Testing data shape (rows, columns):", X_test.shape)

Training data shape (rows, columns): (9600, 34)
Testing data shape (rows, columns): (2400, 34)


In [63]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. Automatically identify which columns are numbers and which are text (categories)
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

# 2. Define the cleaning rules for numerical columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fills blanks with the column's median
    ('scaler', StandardScaler())                   # Standardizes the scale
])

# 3. Define the cleaning rules for categorical columns (if you have any text columns)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')) # <--- This converts text to 0s and 1s
])

# 4. Combine both sets of rules into one master preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing engine built successfully!")

Preprocessing engine built successfully!


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, roc_auc_score
import joblib

# 1. Initialize our three contenders
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

best_model_name = ""
best_roc_auc = 0
best_pipeline = None

print("--- Model Evaluation Results ---\n")

# 2. Train and test each model in a loop
for name, model in models.items():
    # Attach our data cleaning rules to the algorithm
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', model)])
    
    # Train the pipeline on the training data!
    clf.fit(X_train, y_train)
    
    # Make standard predictions (0 or 1) for Recall
    y_pred = clf.predict(X_test)
    
    # Get probabilities (e.g., 0.85 chance of readmission) for ROC-AUC
    # predict_proba gives [chance_of_0, chance_of_1]. We want index 1.
    y_proba = clf.predict_proba(X_test)[:, 1] 
    
    # Calculate how well it did against the hidden test answers
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"{name}:")
    print(f"  Recall:  {recall:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}\n")
    
    # Track the winner based on ROC-AUC
    if roc_auc > best_roc_auc:
        best_roc_auc = roc_auc
        best_model_name = name
        best_pipeline = clf

print(f"Overall Winner: {best_model_name} with ROC-AUC of {best_roc_auc:.4f}")

# 3. Save the winning pipeline for our application backend
joblib.dump(best_pipeline, 'readmission_pipeline.pkl')
print("\nModel successfully saved to disk as 'readmission_pipeline.pkl'!")

In [ ]:
print(X.dtypes)